In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib
from sklearn.compose import ColumnTransformer

In [43]:
df = pd.read_csv('cleaned_students_dropout01.csv')

In [6]:
df

,School_Type,Location,Infrastructure,Teaching_Staff,Gender,Caste,Age,Standard,Socioeconomic_Status,Dropout_Status
0,Government,Semi-Urban,Poor,Poor,Male,SC,14,8,Low,Dropout
1,Private,Rural,Basic,Good,Female,General,15,7,High,Enrolled
2,Government,Rural,Good,Excellent,Male,ST,13,9,Moderate,Dropout
3,Private,Semi-Urban,Basic,Good,Female,OBC,12,10,High,Enrolled
4,Government,Rural,Basic,Poor,Male,SC,16,8,Low,Dropout
...,...,...,...,...,...,...,...,...,...,...
10193,Government,Semi-Urban,Excellent,Good,Male,OBC,13,11,High,Enrolled
10194,Private,Rural,Good,Good,Female,General,10,8,Low,Dropout
10195,Government,Semi-Urban,Good,Good,Male,General,12,10,Moderate,Enrolled
10196,Private,Rural,Basic,Poor,Female,ST,11,9,Low,Dropout


In [8]:
df['Location'].unique()

array(['Semi-Urban', 'Rural', 'Urban'], dtype=object)

In [22]:
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

School_Type: ['Government' 'Private'] unique values
Location: ['Semi-Urban' 'Rural' 'Urban'] unique values
Infrastructure: ['Poor' 'Basic' 'Good' 'Excellent'] unique values
Teaching_Staff: ['Poor' 'Good' 'Excellent' 'Male' 'Female'] unique values
Gender: ['Male' 'Female'] unique values
Caste: ['SC' 'General' 'ST' 'OBC'] unique values
Age: [14 15 13 12 16 10 11] unique values
Standard: [ 8  7  9 10 12  5  6 11] unique values
Socioeconomic_Status: ['Low' 'High' 'Moderate'] unique values
Dropout_Status: ['Dropout' 'Enrolled'] unique values


In [44]:
dfCopy = df.copy()


In [45]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Dropout_Status', axis=1), df['Dropout_Status'], test_size=0.2, random_state=42)

In [17]:
dfCopy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10198 entries, 0 to 10197
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   School_Type           10198 non-null  object
 1   Location              10198 non-null  object
 2   Infrastructure        10198 non-null  object
 3   Teaching_Staff        10198 non-null  object
 4   Gender                10198 non-null  object
 5   Caste                 10198 non-null  object
 6   Age                   10198 non-null  int64 
 7   Standard              10198 non-null  int64 
 8   Socioeconomic_Status  10198 non-null  object
 9   Dropout_Status        10198 non-null  object
dtypes: int64(2), object(8)
memory usage: 796.8+ KB


In [18]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='object')

In [46]:
# cat_col = ['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'gender', 'Caste', 'Socioeconomic_Status']
num_col = ['Age', 'Standard']

In [25]:
Label_cat =['Dropout_Status'] 

In [35]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='object')

In [47]:
onehot_cols = [
    'School_Type',
    'Location',
    'Gender',
    'Caste'
]
ordinal_cols = [
    'Infrastructure',
    'Teaching_Staff',
    'Socioeconomic_Status'
]

In [59]:
ordinal_categories = [
    ['Poor', 'Basic', 'Good', 'Excellent'],  # Infrastructure
    ['Poor', 'Good', 'Excellent', 'Male', 'Female'],  # Teaching_Staff
    ['Low', 'Moderate', 'High']  # Socioeconomic_Status
]

In [38]:
cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

In [60]:
from sklearn.preprocessing import OrdinalEncoder


num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=ordinal_categories))
])
label_pipeline = Pipeline([
    ('label', LabelEncoder())
])


In [61]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_col),
    ('onehot', onehot_pipeline, onehot_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols)
])

In [62]:
from sklearn.linear_model import LogisticRegression


model_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [66]:
model_pipeline.fit(X_train, y_train)
red = model_pipeline.predict(X_test)
print(classification_report(y_test, red))
print(confusion_matrix(y_test, red))


              precision    recall  f1-score   support

     Dropout       0.95      0.94      0.95      1194
    Enrolled       0.92      0.94      0.93       846

    accuracy                           0.94      2040
   macro avg       0.94      0.94      0.94      2040
weighted avg       0.94      0.94      0.94      2040

[[1124   70]
 [  53  793]]


In [72]:
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [ 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1

)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

rf_pred = best_model.predict(X_test)
print('Best Params:', grid.best_params_)
print('Best CV Score:', round(grid.best_score_, 4))
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

Best Params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best CV Score: 0.9966
              precision    recall  f1-score   support

     Dropout       1.00      1.00      1.00      1194
    Enrolled       1.00      1.00      1.00       846

    accuracy                           1.00      2040
   macro avg       1.00      1.00      1.00      2040
weighted avg       1.00      1.00      1.00      2040

[[1193    1]
 [   2  844]]


In [ ]:
# ! pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [73]:
from sklearn.model_selection import cross_val_score
cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')

array([0.99693627, 0.99632353, 0.99754902, 0.99570815, 0.99632128])